# 5. óra — Regresszió: kerékpármegosztás (UCI Bike Sharing, óránkénti)
Ez a notebook **nem vetített anyag** (a slide-ok mindent tartalmaznak), hanem a számok forrása és a hallgatói kísérletezés alapja.
A közös kód a `bike.py`-ban van (adat, jellemzők, időrendi foldok, modellek, mutatók); ugyanezt használja a `results.py` és a `build_assets.py` is.
Futtatás a lecke mappájából: `uv run jupyter lab demo.ipynb`. Az összes cella kb. 1 perc.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import bike as B

df = B.add_leaky_roll(B.add_lags(B.load()))     # lag-ek időbélyeg szerint; a casual/registered már eldobva
raw = pd.read_csv("data/hour.csv")
print("sorok:", len(df), "| hiányzó órák:", int((df.ts.max()-df.ts.min())/pd.Timedelta('1h'))+1-len(df))
print("casual+registered-cnt legnagyobb eltérése:", int((raw.casual+raw.registered-raw.cnt).abs().max()), "-> szivárgás, eldobtuk")
print("fejlesztés:", int((df.ts < B.DEV_END).sum()), "óra | záró teszt (egyszer, a végén):", int((df.ts >= B.FINAL[0]).sum()), "óra")
df.head(3)

sorok: 17379 | hiányzó órák: 165
casual+registered-cnt legnagyobb eltérése: 0 -> szivárgás, eldobtuk
fejlesztés: 15919 óra | záró teszt (egyszer, a végén): 1460 óra


,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,...,cnt,ts,hr_wd,lag1,lag2,lag3,lag24,lag168,roll3,roll3_centered
0,2011-01-01,1,0,1,0,0,6,0,1,0.24,...,16,2011-01-01 00:00:00,0_0,NaN,NaN,NaN,NaN,NaN,NaN,28.000000
1,2011-01-01,1,0,1,1,0,6,0,1,0.22,...,40,2011-01-01 01:00:00,1_0,16.0,NaN,NaN,NaN,NaN,16.0,29.333333
2,2011-01-01,1,0,1,2,0,6,0,1,0.22,...,32,2011-01-01 02:00:00,2_0,40.0,16.0,NaN,NaN,NaN,28.0,28.333333


## 1. Modell-létra, időrendi foldokkal
`B.FOLDS`: kiterjedő ablak, a teszt-blokkok 2012-05–06, 07–08, 09–10; a modell mindig csak a blokk előtti adaton tanul.

In [2]:
cols_fair = B.BASE + ["hr_wd"]
ladder = {
  "átlag": None,
  "lineáris (óra = szám)": B.oof_time(B.lr_naive, B.BASE, df),
  "lineáris (tisztességes)": B.oof_time(B.lr_fair, cols_fair, df),
  "döntési fa": B.oof_time(B.tree, B.BASE, df),
  "random forest": B.oof_time(B.forest, B.BASE, df),
  "xgboost": B.oof_time(B.xgb, B.BASE, df),
}
# múlt heti azonos óra: a naiv szabály, amit meg kell verni
parts = []
for i,(a,b) in enumerate(B.FOLDS):
    tr, te = df.ts < a, (df.ts >= a) & (df.ts < b)
    parts.append(pd.DataFrame({"ts": df.loc[te,"ts"].to_numpy(), "y": df.loc[te,"cnt"].to_numpy(), "p": df.loc[te,"lag168"].fillna(df.loc[tr,"cnt"].mean()).to_numpy(), "fold": i}))
ladder["múlt heti azonos óra"] = pd.concat(parts, ignore_index=True)
ladder.pop("átlag")
tab = pd.DataFrame({k: B.metrics(o.y, o.p) for k, o in ladder.items()}).T.round(3)
tab["fold_rmse"] = [np.round(B.fold_rmse(o),1).tolist() for o in ladder.values()]
tab.sort_values("rmse")

,rmse,mae,wape,rmsle,r2,fold_rmse
xgboost,66.036,45.368,0.161,0.489,0.916,"[62.0, 65.9, 70.1]"
random forest,70.139,44.710,0.159,0.299,0.905,"[69.3, 67.7, 73.4]"
döntési fa,91.894,61.393,0.218,0.367,0.837,"[90.1, 82.7, 102.2]"
múlt heti azonos óra,101.492,58.281,0.207,0.450,0.802,"[99.0, 90.4, 114.1]"
lineáris (tisztességes),110.918,87.119,0.309,0.988,0.763,"[106.9, 106.2, 119.4]"
lineáris (óra = szám),193.165,152.123,0.540,1.259,0.282,"[182.9, 187.3, 208.7]"


## 2. Véletlen vágás vs időrendi, ugyanazokon a teszt-órákon
A véletlen vágás kisebb hibát mutat, mert a tanítóban ott vannak a teszt-órák szomszédai.

In [3]:
from sklearn.model_selection import KFold
dev = df[df.ts < B.DEV_END].reset_index(drop=True)
pr = np.zeros(len(dev))
for tr, te in KFold(3, shuffle=True, random_state=0).split(dev):
    m = B.xgb(); m.fit(dev.loc[tr, B.BASE], dev.loc[tr, "cnt"]); pr[te] = m.predict(dev.loc[te, B.BASE])
sel = (dev.ts >= B.FOLDS[0][0]) & (dev.ts < B.FOLDS[-1][1])
print("véletlen vágás RMSE:", round(B.metrics(dev.cnt[sel], pr[sel.to_numpy()])["rmse"], 1),
      "| időrendi:", round(B.metrics(ladder['xgboost'].y, ladder['xgboost'].p)["rmse"], 1))

véletlen vágás RMSE: 47.3 | időrendi: 66.0


## 3. Diagnosztika az XGBoost out-of-fold jóslatain

In [4]:
ob = ladder["xgboost"].copy(); ob["resid"] = ob.y - ob.p
ob["decilis"] = pd.qcut(ob.p, 10, labels=False, duplicates="drop")
d = ob.groupby("decilis").agg(jósolt=("p","mean"), tényleges=("y","mean"), reziduum_szórás=("resid","std")).round(1)
print(d)                                                   # kalibráció (legfelső decilis) + tölcsér (szórás nő)
a = B.acf(ob.resid.to_numpy(), 48); print("autokorreláció: lag1", round(a[1],2), "lag24", round(a[24],2))
print("negatív jóslatok:", int((ob.p < 0).sum()), "db, a legrosszabb", round(ob.p.min(),1))

             jósolt  tényleges  reziduum_szórás
decilis                                        
0         11.800000       12.1             14.3
1         35.299999       31.1             17.9
2         89.500000       89.8             40.3
3        163.800003      164.4             46.5
4        222.800003      228.5             57.0
5        271.000000      280.7             57.7
6        326.899994      332.8             70.8
7        402.200012      420.5             85.1
8        509.899994      535.0             91.1
9        674.700012      723.0             93.6
autokorreláció: lag1 0.63 lag24 0.32
negatív jóslatok: 44 db, a legrosszabb -37.4


In [5]:
olog = B.oof_time(B.xgb, B.BASE, df, log=True)             # a tölcsér miatt: log(1+cnt)
for nev, o in (("nyers cél", ladder["xgboost"]), ("log(1+cnt)", olog)):
    m = B.metrics(o.y, o.p); print(f"{nev:12s} RMSE {m['rmse']:.1f}  RMSLE {m['rmsle']:.3f}  jósolt/tényleges összeg {o.p.sum()/o.y.sum():.3f}")

nyers cél    RMSE 66.0  RMSLE 0.489  jósolt/tényleges összeg 0.961
log(1+cnt)   RMSE 63.8  RMSLE 0.275  jósolt/tényleges összeg 0.928


## 4. Visszacsatolási kör: először a hibás jellemző, aztán a helyes
A központosított átlag (`roll3_centered`) a mostani és a **következő** órát is tartalmazza: szivárgás.

In [6]:
rows = {
  "alap": B.BASE,
  "+ roll3_centered (SZIVÁRGÁS)": B.BASE + ["roll3_centered"],
  "+ mind a hat késleltetés": B.BASE + B.LAGS,
  "+ csak lag1": B.BASE + ["lag1"],
}
out = {}
for nev, cols in rows.items():
    o = B.oof_time(B.xgb, cols, df); out[nev] = {**B.metrics(o.y, o.p), "fold_rmse": np.round(B.fold_rmse(o),1).tolist()}
pd.DataFrame(out).T[["rmse","mae","wape","r2","fold_rmse"]]

,rmse,mae,wape,r2,fold_rmse
alap,66.036388,45.368237,0.161003,0.916052,"[62.0, 65.9, 70.1]"
+ roll3_centered (SZIVÁRGÁS),27.741854,18.816007,0.066774,0.985184,"[27.1, 26.9, 29.2]"
+ mind a hat késleltetés,50.862904,33.211758,0.117862,0.950198,"[47.9, 49.8, 54.8]"
+ csak lag1,49.151644,32.611268,0.115731,0.953493,"[47.2, 49.2, 50.9]"


In [7]:
o_lr = B.oof_time(lambda: B.lr_fair(B.LAGS), cols_fair + B.LAGS, df)
print("lineáris (tisztességes) + késleltetések RMSE:", round(B.metrics(o_lr.y, o_lr.p)["rmse"], 1), "(a feature engineering itt a lineárisnak is segít)")

lineáris (tisztességes) + késleltetések RMSE: 58.3 (a feature engineering itt a lineárisnak is segít)


## 5. Sodródás: 2011-en tanulunk, 2012 első 10 hónapját jósoljuk

In [8]:
tr, te = df.ts < "2012-01-01", (df.ts >= "2012-01-01") & (df.ts < B.DEV_END)
for nev, mk, cols in (("xgboost alap", B.xgb, B.BASE), ("xgboost + lag1", B.xgb, B.BASE+["lag1"]),
                      ("lineáris alap", B.lr_fair, cols_fair), ("lineáris + késleltetések", lambda: B.lr_fair(B.LAGS), cols_fair+B.LAGS)):
    p, _ = B.fit_predict(mk, cols, tr, te, df); y = df.loc[te, "cnt"].to_numpy()
    print(f"{nev:26s} RMSE {B.metrics(y,p)['rmse']:6.1f}   jósolt/tényleges összeg {p.sum()/y.sum():.2f}")

xgboost alap               RMSE  128.9   jósolt/tényleges összeg 0.63


xgboost + lag1             RMSE   88.8   jósolt/tényleges összeg 0.82
lineáris alap              RMSE  144.4   jósolt/tényleges összeg 0.62
lineáris + késleltetések   RMSE   59.5   jósolt/tényleges összeg 0.94


## 6. Záró teszt (2012-11, 2012-12): egyszer
Szabály, előre rögzítve: a legkisebb CV-RMSE a jelöltek közül -> XGBoost + lag1. A bizonytalanság: napi blokk-bootstrap (az egymás utáni órák nem függetlenek).

In [9]:
trm, tem = df.ts < B.FINAL[0], (df.ts >= B.FINAL[0]) & (df.ts < B.FINAL[1])
p, _ = B.fit_predict(B.xgb, B.BASE + ["lag1"], trm, tem, df)
fin = pd.DataFrame({"ts": df.loc[tem,"ts"].to_numpy(), "y": df.loc[tem,"cnt"].to_numpy(), "p": p})
rm = lambda g: float(np.sqrt(((g.y-g.p)**2).mean()))
bs = B.day_bootstrap(fin, rm)
print("teszt:", {k: round(v,3) for k,v in B.metrics(fin.y, fin.p).items()}, "| RMSE 95%-os intervallum:", np.round(np.percentile(bs,[2.5,97.5]),1))

teszt: {'rmse': 35.362, 'mae': 22.194, 'wape': 0.117, 'rmsle': 0.34, 'r2': 0.959} | RMSE 95%-os intervallum: [30.8 40.1]


In [10]:
# 90%-os kvantilis-sáv
q = {a: B.xgb(objective="reg:quantileerror", quantile_alpha=a).fit(df.loc[trm, B.BASE+["lag1"]], df.loc[trm,"cnt"]).predict(df.loc[tem, B.BASE+["lag1"]]) for a in (0.05, 0.95)}
lo, hi = np.minimum(q[0.05], q[0.95]), np.maximum(q[0.05], q[0.95])
print("lefedettség:", round(float(((fin.y >= lo) & (fin.y <= hi)).mean()), 3), "(névleges 0,90) | átlagos szélesség:", round(float((hi-lo).mean()), 1))

lefedettség: 0.929 (névleges 0,90) | átlagos szélesség: 143.4


A slide-okon szereplő összes szám a `results.py`-ból jön (`results.json`), az ábrák a `build_assets.py`-ból (`assets/`).